<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/lectures_supplementary_code_samples/l11_conversational_ai_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L11 Companion: From Language Models to Conversational AI

This notebook accompanies the L11 deck *From Language Models to Conversational AI*. It follows the lecture's central distinction—**pretraining vs. post-training vs. inference-time control**—plus **evaluation along separate behavioral axes**. All experiments are small, self-contained, and CPU-runnable.

| Section | Lecture Frames Replayed | Stage & Core Question | What You Compute |
|---|---|---|---|
| **N1 · Serialize a Chat** | *Chat Templates; Role Markers in Chat Templates* | **SFT Data Format:** How do messages become one token stream? | Render SFT training sequences vs. generation prefixes across 6 model families. |
| **N2 · Trace Assistant-Only Loss** | *Assistant-Token Loss; Your Turn: Trace the Training Signal* | **SFT Objective:** How do masked context tokens affect training? | Run shifted causal cross-entropy with `ignore_index=-100` and inspect context gradients. |
| **N3 · Compare Two DPO Updates** | *Reading the DPO Loss; Your Turn: Compare Two Updates* | **Preference Alignment:** How does DPO score a policy against a reference? | Compute preferred-to-rejected ratios ($2, 3, 1$), margins $\Delta_\theta$, and losses across $\beta$. |
| **N4 · Change Decoding Scores** | *Inference Controls; Your Turn: Explain the Boundary* | **Inference Control:** What changes when weights $\theta$ stay fixed? | Apply presence and frequency penalties to logits and compare next-token distributions. |
| **N5 · Score Evidence & Calibration** | *Your Turn: Judge Two Responses; Confidence and Calibration* | **Evaluation:** Why separate preference, factuality, and calibration? | Check span-level rubric fidelity on the 98%/72% example and compute Expected Calibration Error (ECE). |

<details>
<summary>Quick formula & notation reference (SFT, DPO, decoding penalties, calibration)</summary>

- **Supervised fine-tuning (SFT) assistant-only loss** over $m$ target assistant tokens $a_1, \dots, a_m$ given serialized context $x$:
  $$\mathcal{L}_{\mathrm{SFT}}(\theta) = -\frac{1}{m}\sum_{t=1}^{m}\log p_\theta(a_t \mid x, a_{<t})$$
- **Bradley–Terry reward-model preference probability and pair loss** for context $x$, preferred response $y^+$, and rejected response $y^-$:
  $$P_\phi(y^+\succ y^-\mid x) = \sigma\bigl(r_\phi(x, y^+) - r_\phi(x, y^-)\bigr), \qquad \ell_{\mathrm{RM}} = -\log P_\phi(y^+\succ y^-\mid x)$$
- **Direct Preference Optimization (DPO) margin and pair loss** relative to frozen reference policy $p_{\mathrm{ref}}$ with scale coefficient $\beta > 0$:
  $$\Delta_\theta = \log\frac{p_\theta(y^+\mid x)}{p_\theta(y^-\mid x)} - \log\frac{p_{\mathrm{ref}}(y^+\mid x)}{p_{\mathrm{ref}}(y^-\mid x)}, \qquad \ell(\theta) = -\log\sigma(\beta\cdot\Delta_\theta), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$
  *(Jurafsky & Martin Ch. 8 notation: $\pi_\theta, \pi_{\mathrm{ref}}, o_w, o_l$ correspond to $p_\theta, p_{\mathrm{ref}}, y^+, y^-$.)*
- **Reference-regularized RL objective** (Appendix):
  $$\max_\theta\;\mathbb{E}_{x}\left[\mathbb{E}_{y\sim p_\theta(\cdot\mid x)}[r_\phi(x,y)] - \beta\cdot D_{\mathrm{KL}}\!\left(p_\theta(\cdot\mid x)\,\|\,p_{\mathrm{ref}}(\cdot\mid x)\right)\right]$$
- **Illustrative repetition penalties** on raw logit $z_v$ given observed token count $c_v$:
  $$z'_v = z_v - \alpha_{\mathrm{pres}}\cdot\mathbb{I}[c_v > 0] - \alpha_{\mathrm{freq}}\cdot c_v$$
- **Expected Calibration Error (ECE)** across $B$ confidence bins with count $n_b$, mean predicted confidence $\hat{p}_b$, and empirical accuracy $\mathrm{acc}_b$:
  $$\mathrm{ECE} = \sum_{b=1}^{B} \frac{n_b}{N}\cdot\bigl|\hat{p}_b - \mathrm{acc}_b\bigr|$$
</details>

- **Core (approx. 35–45 minutes):** N1–N5.


In [1]:
# Dependencies: Python, PyTorch, NumPy.
# Only Colab installs missing packages.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "torch>=2.0" "numpy>=1.24"

import importlib.metadata
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=4, sci_mode=False)
np.random.seed(42)
torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
for package in ("torch", "numpy"):
    print(f"{package}: {importlib.metadata.version(package)}")


Device: cpu
torch: 2.9.0
numpy: 1.26.4


## N1. Serialize a Chat

**Roles and turn boundaries are serialized into a single token sequence; generation starts right after the open assistant header.**

Replays: *Chat Templates; Role Markers in Chat Templates*.

A causal decoder consumes a flat sequence of token IDs, not a list of Python dictionaries. A **chat template** serializes role labels (`system`, `user`, `assistant`) and turn boundaries into text or special tokens.

Consider the overfitting exchange from the lecture slides:
- **System:** `Explain concepts briefly.`
- **User:** `What is overfitting?`
- **Assistant:** `A model fits training-specific patterns and generalizes poorly.`

When building a **training sequence** for supervised fine-tuning (SFT), the template serializes all turns, including the assistant response and its closing turn token. When building a **generation prefix** at inference time (`add_generation_prompt=True`), the template stops at the point where the assistant should begin generating tokens.

In the code cell below, inspect the schematic template and the six model families from the appendix (*Role Markers in Chat Templates*: `Llama 3`, `Qwen 2.5 (ChatML)`, `Mistral 7B v0.1`, `Gemma 2`, `Phi-3`, `gpt-oss`). Look at where the caller's instruction (`"Explain concepts briefly."`) is placed and what marker ends each generation prefix.

**Predict:**
1. In the schematic template, what exact substring marks the end of the generation prefix where the causal decoder begins sampling the assistant's first token?
2. Looking at `render_family_prefix` below, which model families do *not* emit a dedicated `system` role block for the caller's instruction, and where does that instruction go instead?


In [2]:
messages = [
    {"role": "system", "content": "Explain concepts briefly."},
    {"role": "user", "content": "What is overfitting?"},
    {"role": "assistant", "content": "A model fits training-specific patterns and generalizes poorly."},
]

def render_schematic(msgs, add_generation_prompt=False):
    parts = []
    for m in msgs:
        if add_generation_prompt and m["role"] == "assistant":
            continue
        parts.append(f"<{m['role']}> {m['content']} <end>")
    if add_generation_prompt:
        parts.append("<assistant>")
    return "\n".join(parts)

print("=== Schematic Template (Training Sequence) ===")
train_str = render_schematic(messages, add_generation_prompt=False)
print(train_str)

print("\n=== Schematic Template (Generation Prefix) ===")
gen_prefix = render_schematic(messages[:2], add_generation_prompt=True)
print(gen_prefix)
print(f"\nContinuation added during generation: {train_str[len(gen_prefix):]!r}")

# Compare generation prefixes across model families from the appendix table
family_specs = [
    (
        "Llama 3",
        "Dedicated system header",
        "<|start_header_id|>assistant<|end_header_id|>\n\n",
        lambda sys_m, usr_m: (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{sys_m}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n{usr_m}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        ),
    ),
    (
        "Qwen 2.5 (ChatML)",
        "Dedicated system header",
        "<|im_start|>assistant\n",
        lambda sys_m, usr_m: (
            f"<|im_start|>system\n{sys_m}<|im_end|>\n"
            f"<|im_start|>user\n{usr_m}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        ),
    ),
    (
        "Mistral 7B v0.1",
        "Merged into first user [INST]",
        "[/INST]",
        lambda sys_m, usr_m: f"<s>[INST] {sys_m}\n\n{usr_m} [/INST]",
    ),
    (
        "Gemma 2 (user only)",
        "Rejected (no system role)",
        "<start_of_turn>model\n",
        lambda sys_m, usr_m: (
            f"<bos><start_of_turn>user\n{usr_m}<end_of_turn>\n"
            f"<start_of_turn>model\n"
        ),
    ),
    (
        "Phi-3",
        "Dedicated <|system|> tag",
        "<|assistant|>\n",
        lambda sys_m, usr_m: f"<|system|>\n{sys_m}<|end|>\n<|user|>\n{usr_m}<|end|>\n<|assistant|>\n",
    ),
    (
        "gpt-oss",
        "Moved to developer role",
        "<|start|>assistant",
        lambda sys_m, usr_m: (
            f"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\n"
            f"Reasoning: medium\n# Valid channels: analysis, commentary, final.<|end|>"
            f"<|start|>developer<|message|># Instructions\n\n{sys_m}<|end|>"
            f"<|start|>user<|message|>{usr_m}<|end|>"
            f"<|start|>assistant"
        ),
    ),
]

print("\n=== Compact Comparison Across Model Families (Appendix) ===")
print(f"{'Family':<22} {'Caller System Instruction':<30} {'Generation Prefix Ends With'}")
print("-" * 92)
for fam, sys_handling, suffix, _ in family_specs:
    print(f"{fam:<22} {sys_handling:<30} {suffix!r}")

print("\n=== Full Rendered Generation Prefixes ===")
for fam, _, _, renderer in family_specs:
    print(f"\n--- {fam} ---")
    print(renderer(messages[0]["content"], messages[1]["content"]))


=== Schematic Template (Training Sequence) ===
<system> Explain concepts briefly. <end>
<user> What is overfitting? <end>
<assistant> A model fits training-specific patterns and generalizes poorly. <end>

=== Schematic Template (Generation Prefix) ===
<system> Explain concepts briefly. <end>
<user> What is overfitting? <end>
<assistant>

Continuation added during generation: ' A model fits training-specific patterns and generalizes poorly. <end>'

=== Compact Comparison Across Model Families (Appendix) ===
Family                 Caller System Instruction      Generation Prefix Ends With
--------------------------------------------------------------------------------------------
Llama 3                Dedicated system header        '<|start_header_id|>assistant<|end_header_id|>\n\n'
Qwen 2.5 (ChatML)      Dedicated system header        '<|im_start|>assistant\n'
Mistral 7B v0.1        Merged into first user [INST]  '[/INST]'
Gemma 2 (user only)    Rejected (no system role)      '<start_o

**Compare with your prediction:** Check the `Continuation added during generation` line and the `Compact Comparison Across Model Families` table above against your predictions before expanding the summary.

<details>
<summary>Summary of results: Generation prefixes and system-role handling across the six families</summary>

1. **Where generation starts:** In the schematic format, the generation prefix ends right after `<assistant>`. The remaining string (`' A model fits training-specific patterns and generalizes poorly. <end>'`) is the target continuation during SFT and the generated output at inference time.
2. **System role differences:**
   - `Llama 3`, `Qwen 2.5 (ChatML)`, and `Phi-3` place the caller's instruction inside a dedicated `system` block.
   - `Mistral 7B v0.1` has no system marker and merges the instruction into the first user turn inside `[INST] ... [/INST]`.
   - `Gemma 2` rejects a `system` role altogether and calls the assistant role `model` (`<start_of_turn>model\n`).
   - `gpt-oss` reserves `system` for a provider header (declaring valid channels `analysis`, `commentary`, `final`) and places the caller's system instruction inside a `developer` turn (`<|start|>developer<|message|>...`).
</details>

**Check yourself.**

1. Why does passing a raw string without the model's trained chat template often degrade an instruction-tuned model's response quality?
2. Why is a `<system>` or `<|start|>developer` token a learned cue rather than a security barrier?
3. What happens at inference time if a generation prefix forgets to append the open assistant header (e.g., stops at `<|eot_id|>` after the user message)?

<details>
<summary>Solution: Template mismatch, learned role cues, and missing generation prompts</summary>

1. During SFT and preference training, assistant responses always follow specific role and boundary tokens. Without those tokens, the input looks like an unformatted pretraining prefix, so the model often continues the user's question or mimics document structure instead of answering.
2. All tokens—whether from `system`, `developer`, or `user`—enter the same embedding matrix and self-attention layers. Post-training teaches the model a statistical tendency to prioritize higher-ranked roles, but conflicting instructions inside the `user` turn can still override them (prompt injection).
3. If the prefix stops right after the user's end-of-turn token without opening the assistant turn, the model has to predict which role speaks next; it may open another `<user>` turn or emit an end-of-sequence token immediately.
</details>


## N2. Trace Assistant-Only Loss

**In assistant-only SFT, the target mask (`ignore_index = -100`) selects which token predictions add to the loss, while causal attention controls which tokens condition those predictions.**

Replays: *Assistant-Token Loss; Your Turn: Trace the Training Signal*.

In standard SFT, the serialized sequence is split into context tokens $x$ (system, user, and the opening `<assistant>` marker) and $m$ supervised assistant target tokens $a_1, \dots, a_m$ (including the closing `<end>` token):
$$\mathcal{L}_{\mathrm{SFT}}(\theta) = -\frac{1}{m}\sum_{t=1}^{m}\log p_\theta(a_t \mid x, a_{<t})$$

In PyTorch, shifted causal cross-entropy uses `ignore_index = -100` for positions with target mask $0$:
- **Target mask** selects which predicted positions contribute a direct loss term to $\mathcal{L}_{\mathrm{SFT}}(\theta)$.
- **Causal attention** controls which earlier tokens are visible when computing representations.

We now test the exact question from *Your Turn: Trace the Training Signal*:
> *The user tokens have target mask 0. Replace `"overfitting"` by `"underfitting"` but keep the assistant target unchanged. Can the loss change?*

**Predict:**
1. In a sequence of $9$ context tokens followed by $5$ assistant tokens ($14$ tokens total, $13$ shifted next-token pairs), how many positions have target mask $1$ (`Supervised = YES`), and which input token sits at the position that predicts the first assistant word $a_1$ (`"Fits"`)?
2. When we replace `"overfitting?"` with `"underfitting?"` in the user turn (which has target mask $0$), will $\mathcal{L}_{\mathrm{SFT}}(\theta)$ stay identical or change? Will the gradient on the masked user token's embedding be zero or non-zero?


In [3]:
# Word-level vocabulary for the toy SFT sequence
vocab = [
    "<system>", "Explain", "briefly.", "<end>",
    "<user>", "What", "is", "overfitting?", "underfitting?",
    "<assistant>", "Fits", "training", "noise", "poorly."
]
stoi = {w: i for i, w in enumerate(vocab)}

context_over = ["<system>", "Explain", "briefly.", "<end>", "<user>", "What", "is", "overfitting?", "<assistant>"]
context_under = ["<system>", "Explain", "briefly.", "<end>", "<user>", "What", "is", "underfitting?", "<assistant>"]
assistant_target = ["Fits", "training", "noise", "poorly.", "<end>"]

def build_sft_tensors(context_tokens, answer_tokens):
    full_tokens = context_tokens + answer_tokens
    ids = torch.tensor([stoi[w] for w in full_tokens], dtype=torch.long)
    # Shifted causal pairs: input_ids[t] predicts target_ids[t] = full_ids[t+1]
    input_ids = ids[:-1]
    target_ids = ids[1:].clone()
    # Context length is len(context_tokens). The last context token ('<assistant>' at index len(context)-1)
    # predicts the first assistant token a_1 (at full_ids[len(context)] -> target_ids[len(context)-1]).
    # Therefore target positions 0 .. len(context)-2 predict context tokens and get masked to -100.
    n_masked = len(context_tokens) - 1
    target_ids[:n_masked] = -100
    return full_tokens, input_ids, target_ids

full_over, in_over, tgt_over = build_sft_tensors(context_over, assistant_target)
full_under, in_under, tgt_under = build_sft_tensors(context_under, assistant_target)

print("Shifted supervision alignment (overfitting example):")
print(f"{'Pos':<4} {'Input token':<15} {'Next token':<15} {'Label':<8} {'Supervised?'}")
print("-" * 58)
for t in range(len(in_over)):
    inp_w = full_over[t]
    nxt_w = full_over[t + 1]
    lbl = tgt_over[t].item()
    sup = "YES (mask 1)" if lbl != -100 else "no  (mask 0)"
    print(f"{t:<4} {inp_w:<15} {nxt_w:<15} {lbl:<8} {sup}")

# Tiny causal self-attention LM to trace loss and gradients
class TinyCausalLM(nn.Module):
    def __init__(self, vocab_size, d_model=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        # x: [T]
        h = self.embed(x)  # [T, d]
        Q, K, V = self.W_Q(h), self.W_K(h), self.W_V(h)
        scores = (Q @ K.T) / (h.size(-1) ** 0.5)
        causal_mask = torch.triu(torch.full_like(scores, float("-inf")), diagonal=1)
        attn = F.softmax(scores + causal_mask, dim=-1)
        ctx = attn @ V
        logits = self.head(h + ctx)  # [T, |V|]
        return logits

torch.manual_seed(7)
tiny_lm = TinyCausalLM(len(vocab), d_model=16)

# 1. Forward + backward with "overfitting?" in context
tiny_lm.zero_grad()
logits_over = tiny_lm(in_over)
per_pos_loss_over = F.cross_entropy(logits_over, tgt_over, ignore_index=-100, reduction="none")
loss_over = F.cross_entropy(logits_over, tgt_over, ignore_index=-100, reduction="mean")
loss_over.backward()
grad_over_tok = tiny_lm.embed.weight.grad[stoi["overfitting?"]].norm().item()

# 2. Forward + backward with "underfitting?" in context (same assistant target & mask)
tiny_lm.zero_grad()
logits_under = tiny_lm(in_under)
per_pos_loss_under = F.cross_entropy(logits_under, tgt_under, ignore_index=-100, reduction="none")
loss_under = F.cross_entropy(logits_under, tgt_under, ignore_index=-100, reduction="mean")
loss_under.backward()
grad_under_tok = tiny_lm.embed.weight.grad[stoi["underfitting?"]].norm().item()

print("\nPer-position losses (overfitting vs. underfitting context):")
for t in range(len(in_over)):
    print(
        f"  Pos {t:2d} (predicts {full_over[t+1]:<13}): "
        f"loss_over = {per_pos_loss_over[t].item():.4f} | "
        f"loss_under = {per_pos_loss_under[t].item():.4f}"
    )

print(f"\nMean SFT loss with 'overfitting?':  {loss_over.item():.4f}")
print(f"Mean SFT loss with 'underfitting?': {loss_under.item():.4f}")
print(f"Absolute difference in SFT loss:    {abs(loss_over.item() - loss_under.item()):.4f}")
print(f"Grad norm on masked context token 'overfitting?' embedding:  {grad_over_tok:.4f}")
print(f"Grad norm on masked context token 'underfitting?' embedding: {grad_under_tok:.4f}")


Shifted supervision alignment (overfitting example):
Pos  Input token     Next token      Label    Supervised?
----------------------------------------------------------
0    <system>        Explain         -100     no  (mask 0)
1    Explain         briefly.        -100     no  (mask 0)
2    briefly.        <end>           -100     no  (mask 0)
3    <end>           <user>          -100     no  (mask 0)
4    <user>          What            -100     no  (mask 0)
5    What            is              -100     no  (mask 0)
6    is              overfitting?    -100     no  (mask 0)
7    overfitting?    <assistant>     -100     no  (mask 0)
8    <assistant>     Fits            10       YES (mask 1)
9    Fits            training        11       YES (mask 1)
10   training        noise           12       YES (mask 1)
11   noise           poorly.         13       YES (mask 1)
12   poorly.         <end>           3        YES (mask 1)

Per-position losses (overfitting vs. underfitting context):
  

**Compare with your prediction:** Inspect the `Per-position losses` for positions `0..7` versus `8..12`, the two mean SFT losses, and the gradient norms at the bottom of the output.

<details>
<summary>Summary of results: Shifted target alignment, loss shift (2.5489 vs. 2.5774), and context embedding gradients</summary>

1. **Zero direct context loss:** Positions `0` through `7` predict context tokens (`Explain` through `<assistant>`); because their target labels are `-100`, their direct loss terms are `0.0000` in both runs.
2. **Shifted boundary at position 8:** Position `8` has input token `<assistant>` and predicts the first assistant word `Fits` ($a_1$). Positions `8` through `12` provide the $m = 5$ supervised assistant loss terms whose average is $\mathcal{L}_{\mathrm{SFT}}(\theta)$.
3. **Context influence without direct context loss:** Replacing `"overfitting?"` with `"underfitting?"` at position `7` alters the attention keys and values seen by every subsequent assistant position `8..12`, shifting the mean assistant loss from `2.5489` to `2.5774` (difference `0.0285`) and propagating a non-zero gradient (`0.0111` / `0.0133`) back into the masked context token's embedding vector.
</details>

**Check yourself.**

1. Why does the last context token (`<assistant>` at position 8) carry a supervised target label instead of `-100`?
2. Through which mathematical path in the network does a masked context token receive a non-zero parameter gradient during SFT?
3. What is the difference between setting a token's target mask to `0` (`-100`) and masking that token out in the causal attention matrix?

<details>
<summary>Solution: Shifted prediction alignment, gradient flow through attention, and target vs. attention masks</summary>

1. In next-token prediction, input position $t$ predicts token $t+1$. The `<assistant>` delimiter at position 8 is the conditioning prefix position whose next token is the first assistant word $a_1$ (`Fits`). Thus position 8's target is $a_1$, which must be supervised.
2. Each supervised assistant position $t \ge 8$ attends to earlier context keys and values $K_j, V_j$ ($j \le 8$). The loss $\ell_t$ depends on the attention output $\sum_{j \le t} A_{tj} V_j$, which depends on the context representations $h_j$ and their embeddings. By the chain rule, $\partial \mathcal{L}_{\mathrm{SFT}} / \partial e_{\text{context}} \ne 0$.
3. The **target mask** omits direct loss terms $\ell_j$ for context predictions while leaving context tokens fully visible to later queries. Masking a context token in the **attention matrix** would prevent the assistant tokens from seeing the user's question at all.
</details>


## N3. Compare Two DPO Updates

**The DPO loss measures how a policy changes the preferred-to-rejected likelihood ratio relative to a frozen reference policy.**

Replays: *Reading the DPO Loss; Your Turn: Compare Two Updates*.

For context $x$, preferred response $y^+$, and rejected response $y^-$, Direct Preference Optimization (DPO) defines the reference-relative log-ratio margin:
$$\Delta_\theta = \log\frac{p_\theta(y^+\mid x)}{p_\theta(y^-\mid x)} - \log\frac{p_{\mathrm{ref}}(y^+\mid x)}{p_{\mathrm{ref}}(y^-\mid x)}$$
and minimizes the pair loss:
$$\ell(\theta) = -\log\sigma(\beta\cdot\Delta_\theta), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}, \quad \beta > 0$$

Consider the exact probabilities from *Your Turn: Compare Two Updates*:
- **Frozen reference $p_{\mathrm{ref}}$:** $p_{\mathrm{ref}}(y^+\mid x) = 0.20$, $p_{\mathrm{ref}}(y^-\mid x) = 0.10$ (preferred-to-rejected ratio $0.20/0.10 = 2$).
- **Policy A:** $p_A(y^+\mid x) = 0.30$, $p_A(y^-\mid x) = 0.10$.
- **Policy B:** $p_B(y^+\mid x) = 0.20$, $p_B(y^-\mid x) = 0.20$.

**Predict:**
1. What are the preferred-to-rejected probability ratios $p(y^+\mid x)/p(y^-\mid x)$ for Policy A and Policy B relative to the reference ratio of $2$?
2. At the reference policy ($\Delta_{\mathrm{ref}} = 0$), what is $\ell_{\mathrm{ref}}$? For $\beta = 1$ (and for $\beta = 0.5$, as in the *Further Study* slide), is each policy's loss **above** or **below** the reference loss?
3. Policy B keeps $p_B(y^+\mid x) = 0.20$ equal to the reference probability. Does keeping $p(y^+\mid x)$ fixed keep the DPO loss unchanged?


In [4]:
policies = {
    "Reference (frozen)": (0.20, 0.10),
    "Policy A":           (0.30, 0.10),
    "Policy B":           (0.20, 0.20),
}

p_ref_pos, p_ref_neg = policies["Reference (frozen)"]
ref_ratio = p_ref_pos / p_ref_neg
ref_log_ratio = np.log(ref_ratio)

def dpo_metrics(p_pos, p_neg, beta=1.0):
    ratio = p_pos / p_neg
    delta = np.log(ratio) - ref_log_ratio
    prob_pref = 1.0 / (1.0 + np.exp(-beta * delta))
    loss = -np.log(prob_pref)
    return ratio, delta, prob_pref, loss

print(f"{'Policy':<20} {'p(y+|x)':<9} {'p(y-|x)':<9} {'Ratio':<7} {'Delta':<9} {'sigma(Delta)':<14} {'Loss (beta=1)'}")
print("-" * 84)
for name, (p_pos, p_neg) in policies.items():
    ratio, delta, sig, loss = dpo_metrics(p_pos, p_neg, beta=1.0)
    print(f"{name:<20} {p_pos:<9.2f} {p_neg:<9.2f} {ratio:<7.2f} {delta:<+9.4f} {sig:<14.4f} {loss:.4f}")

# Sweep beta across typical values (0.01 to 1.0) as noted on the slide
betas = [0.01, 0.05, 0.10, 0.50, 1.00]
print("\nEffect of beta on DPO loss:")
print(f"{'beta':<8} {'Loss(Ref)':<12} {'Loss(Policy A)':<16} {'Loss(Policy B)':<16}")
print("-" * 52)
for b in betas:
    _, _, _, l_ref = dpo_metrics(0.20, 0.10, beta=b)
    _, _, _, l_a = dpo_metrics(0.30, 0.10, beta=b)
    _, _, _, l_b = dpo_metrics(0.20, 0.20, beta=b)
    print(f"{b:<8.2f} {l_ref:<12.4f} {l_a:<16.4f} {l_b:<16.4f}")

# --- Your Turn Starter: Test a Policy D that lowers p(y-|x) without raising p(y+|x) ---
# TODO: Pick p_D(y+|x) = 0.20 and p_D(y-|x) = 0.05 (so p(y+|x) is unchanged, but p(y-|x) is halved).
# Predict the ratio, Delta_D, and whether Loss(Policy D) falls below Policy A's loss (0.5108).
p_d_pos, p_d_neg = 0.20, 0.05
ratio_d, delta_d, sig_d, loss_d = dpo_metrics(p_d_pos, p_d_neg, beta=1.0)
print(f"\nYour Turn Check — Policy D ({p_d_pos:.2f}, {p_d_neg:.2f}): ratio={ratio_d:.2f}, Delta={delta_d:+.4f}, loss={loss_d:.4f}")


Policy               p(y+|x)   p(y-|x)   Ratio   Delta     sigma(Delta)   Loss (beta=1)
------------------------------------------------------------------------------------
Reference (frozen)   0.20      0.10      2.00    +0.0000   0.5000         0.6931
Policy A             0.30      0.10      3.00    +0.4055   0.6000         0.5108
Policy B             0.20      0.20      1.00    -0.6931   0.3333         1.0986

Effect of beta on DPO loss:
beta     Loss(Ref)    Loss(Policy A)   Loss(Policy B)  
----------------------------------------------------
0.01     0.6931       0.6911           0.6966          
0.05     0.6931       0.6831           0.7106          
0.10     0.6931       0.6731           0.7284          
0.50     0.6931       0.5969           0.8814          
1.00     0.6931       0.5108           1.0986          

Your Turn Check — Policy D (0.20, 0.05): ratio=4.00, Delta=+0.6931, loss=0.4055


**Compare with your prediction:** Check the `Ratio`, `Delta`, and `Loss (beta=1)` columns in the table above against the slide *Your Turn: Compare Two Updates* before opening the summary.

<details>
<summary>Summary of results: Reference, Policy A, and Policy B ratios (2, 3, 1), margins, and losses across β</summary>

1. **Three ratios:** The preferred-to-rejected ratios are $2.00$ for the reference ($0.20/0.10$), $3.00$ for Policy A ($0.30/0.10$), and $1.00$ for Policy B ($0.20/0.20$).
2. **Reference baseline:** At the reference policy, $\Delta_{\mathrm{ref}} = 0$, $\sigma(0) = 0.5$, and the loss is $\log 2 \approx 0.6931$ for every $\beta$.
3. **Policy A falls below the reference loss:** Raising the ratio from $2$ to $3$ gives a positive margin $\Delta_A = \log(3) - \log(2) = \log(1.5) \approx +0.4055$ and lowers the loss at $\beta=1$ to $0.5108 \approx 0.511$.
4. **Policy B rises above the reference loss:** Even though $p_B(y^+\mid x) = 0.20$ is unchanged, doubling $p_B(y^-\mid x)$ from $0.10$ to $0.20$ cuts the ratio from $2$ to $1$. That gives a negative margin $\Delta_B = \log(1) - \log(2) = \log(0.5) \approx -0.6931$ and raises the loss at $\beta=1$ to $1.0986 \approx 1.099$.
5. **Your Turn starter (Policy D):** Halving $p_D(y^-\mid x)$ to $0.05$ while keeping $p_D(y^+\mid x) = 0.20$ raises the ratio to $4.00$ ($\Delta_D = \log(4) - \log(2) = \log 2 \approx +0.6931$), achieving an even lower loss (`0.4055`) than Policy A (`0.5108`).
6. **Effect of $\beta$ (including the *Further Study* check at $\beta=0.5$):** At $\beta = 0.5$, both losses move closer to the reference loss $\ell_{\mathrm{ref}} = \log 2 \approx 0.6931$ ($\ell_A = 0.5969$ vs. $0.5108$ at $\beta=1$; $\ell_B = 0.8814$ vs. $1.0986$ at $\beta=1$). At typical practical values ($\beta \in [0.01, 0.10]$), the same margins $\Delta_A, \Delta_B$ produce even smaller deviations around $0.6931$.
</details>

**Check yourself.**

1. Suppose Policy C assigns $p_C(y^+\mid x) = 0.10$ and $p_C(y^-\mid x) = 0.05$. Even though $p_C(y^+\mid x)$ dropped from $0.20$ to $0.10$, what is Policy C's DPO margin $\Delta_C$ and loss relative to the reference?
2. Why does standard offline DPO require two models in memory during training instead of four as in standard PPO-based RLHF?
3. What is the role of $\beta$ in the underlying RL objective $\max_\theta \mathbb{E}_{x}\bigl[\mathbb{E}_{y\sim p_\theta(\cdot\mid x)}[r_\phi(x,y)] - \beta\cdot D_{\mathrm{KL}}(p_\theta(\cdot\mid x) \| p_{\mathrm{ref}}(\cdot\mid x))\bigr]$?

<details>
<summary>Solution: Ratio invariance, memory footprint (2 vs. 4 models), and the role of beta</summary>

1. For Policy C, the preferred-to-rejected ratio is $0.10 / 0.05 = 2.0$, identical to the reference ratio $0.20 / 0.10 = 2.0$. Thus $\Delta_C = \log(2) - \log(2) = 0$ and $\ell_C = \log 2 \approx 0.6931$. DPO compares the *relative* ratio of $y^+$ to $y^-$ against the reference ratio, not the absolute likelihood of $y^+$ in isolation.
2. Standard PPO-based RLHF keeps four models in memory: the trainable policy, the frozen reference policy, a separate reward model, and a value/critic network. Standard offline DPO reparameterizes the objective so the reward is expressed implicitly via $\beta\cdot\log(p_\theta / p_{\mathrm{ref}})$, requiring only the trainable policy and the frozen reference policy.
3. For a fixed reward scale, $\beta > 0$ controls the strength of the KL divergence penalty against drifting away from the frozen reference policy $p_{\mathrm{ref}}$; in the DPO loss it scales the log-ratio margin $\Delta_\theta$ inside $\sigma(\beta\cdot\Delta_\theta)$.
</details>


## N4. Change Decoding Scores

**Inference controls alter next-token scores or spend extra forward passes while the trained weights remain completely fixed.**

Replays: *Inference Controls; Your Turn: Explain the Boundary*.

At inference time, no gradients are computed and no optimizer step modifies $\theta$. Instead, decoding rules transform the logits $\mathbf{z} \in \mathbb{R}^{|V|}$ emitted at the current step before sampling or selecting the next token.

Two common repetition controls modify logits using the token counts $c_v \in \{0, 1, 2, \dots\}$ already observed in the context or generated response:
- **Illustrative presence penalty ($\alpha_{\mathrm{pres}}$):** subtracts a flat penalty if token $v$ has appeared at least once:
  $$z'_v = z_v - \alpha_{\mathrm{pres}} \cdot \mathbb{I}[c_v > 0]$$
- **Illustrative frequency penalty ($\alpha_{\mathrm{freq}}$):** subtracts a penalty proportional to the token's count $c_v$:
  $$z'_v = z_v - \alpha_{\mathrm{freq}} \cdot c_v$$

*(Exact formulas vary slightly across API providers, but the distinction is always binary occurrence vs. occurrence count.)*

Consider five candidate tokens with identical raw logit $z_v = 2.0$, differing only in their history counts in the current turn:
- `"overfitting"`: count $c = 4$ (repeated four times)
- `"model"`: count $c = 1$ (appeared once)
- `"training"`: count $c = 1$ (appeared once)
- `"generalizes"`: count $c = 0$ (not seen yet)
- `"test"`: count $c = 0$ (not seen yet)

**Predict:**
1. Under a **presence penalty** ($\alpha_{\mathrm{pres}} = 1.2$, $\alpha_{\mathrm{freq}} = 0$), will `"overfitting"` ($c=4$) and `"model"` ($c=1$) receive different adjusted logits or the exact same adjusted logit?
2. Under a **frequency penalty** ($\alpha_{\mathrm{pres}} = 0$, $\alpha_{\mathrm{freq}} = 0.6$), which token's probability drops the most, and how much probability remains on `"model"` ($c=1$)?
3. Does running 2,000 intermediate reasoning steps with these penalties update any model parameter in $\theta$?


In [5]:
candidates = ["overfitting", "model", "training", "generalizes", "test"]
raw_logits = torch.tensor([2.0, 2.0, 2.0, 2.0, 2.0])
counts     = torch.tensor([4,   1,   1,   0,   0], dtype=torch.float32)

def apply_penalties(logits, token_counts, alpha_pres=0.0, alpha_freq=0.0):
    presence_indicator = (token_counts > 0).float()
    adjusted = logits - alpha_pres * presence_indicator - alpha_freq * token_counts
    probs = F.softmax(adjusted, dim=-1)
    return adjusted, probs

_, probs_base = apply_penalties(raw_logits, counts, 0.0, 0.0)
logits_pres, probs_pres = apply_penalties(raw_logits, counts, alpha_pres=1.2, alpha_freq=0.0)
logits_freq, probs_freq = apply_penalties(raw_logits, counts, alpha_pres=0.0, alpha_freq=0.6)

print(f"{'Token':<14} {'Count':<6} {'Base P':<9} {'Pres Logit':<11} {'Pres P':<9} {'Freq Logit':<11} {'Freq P':<9}")
print("-" * 74)
for i, tok in enumerate(candidates):
    print(
        f"{tok:<14} {int(counts[i]):<6d} "
        f"{probs_base[i].item():<9.3f} "
        f"{logits_pres[i].item():<11.2f} {probs_pres[i].item():<9.3f} "
        f"{logits_freq[i].item():<11.2f} {probs_freq[i].item():<9.3f}"
    )


Token          Count  Base P    Pres Logit  Pres P    Freq Logit  Freq P   
--------------------------------------------------------------------------
overfitting    4      0.200     0.80        0.104     -0.40       0.028    
model          1      0.200     0.80        0.104     1.40        0.172    
training       1      0.200     0.80        0.104     1.40        0.172    
generalizes    0      0.200     2.00        0.344     2.00        0.314    
test           0      0.200     2.00        0.344     2.00        0.314    


**Compare with your prediction:** Compare `Pres Logit` / `Pres P` against `Freq Logit` / `Freq P` for `"overfitting"` ($c=4$) and `"model"` ($c=1$).

<details>
<summary>Summary of results: Presence (0.104) vs. frequency (0.028 vs. 0.172) probability shifts</summary>

1. **Baseline:** Because all five raw logits are `2.0`, every candidate starts with probability `0.200`.
2. **Presence penalty ($\alpha_{\mathrm{pres}} = 1.2$):** `"overfitting"` ($c=4$), `"model"` ($c=1$), and `"training"` ($c=1$) all have $\mathbb{I}[c_v > 0] = 1$, so all three drop to the exact same logit `0.80` and probability `0.104`. Unseen tokens (`"generalizes"`, `"test"`) stay at logit `2.00` and rise to probability `0.344` each.
3. **Frequency penalty ($\alpha_{\mathrm{freq}} = 0.6$):** The penalty scales with $c_v$. `"overfitting"` ($c=4$) loses $4 \cdot 0.6 = 2.4$ logit points (down to `-0.40`, probability `0.028`), whereas `"model"` and `"training"` ($c=1$) lose only $0.6$ (logit `1.40`, probability `0.172`), and unseen tokens have probability `0.314` each.
</details>

**Check yourself.**

1. When would you prefer a moderate frequency penalty over a large presence penalty in long multi-paragraph explanations?
2. Revisit *Your Turn: Explain the Boundary*: when an assistant generates 2,000 intermediate reasoning tokens before answering, what state *does* change at each step, and what evidence would be required to claim that its weights changed?

<details>
<summary>Solution: Presence vs. frequency tradeoffs and the training-inference boundary</summary>

1. In a technical explanation of overfitting, core domain terms like `"model"` or `"training"` naturally need to appear more than once. A large presence penalty penalizes the second mention just as heavily as the tenth, encouraging awkward synonyms; a frequency penalty gently discourages runaway loops ($c=4, 5, \dots$) while tolerating a second mention.
2. During the 2,000 intermediate tokens, the conditioning prefix, hidden activations, and KV-cache grow at every forward pass, and decoding controls transform the output logits. None of this modifies the stored parameter tensors $\theta$. Establishing that weights changed requires a loss/reward computation, backward pass (or gradient estimate), and an optimizer update step that alters $\theta$.
</details>


## N5. Score Evidence and Check Calibration

**A reference-based rubric catches factual and constraint errors hidden by confident style, while calibration checks whether numeric confidence matches empirical accuracy.**

Replays: *Your Turn: Judge Two Responses; Confidence and Calibration*.

In *Your Turn: Judge Two Responses*, we evaluate two fluent responses against an explicit reference and request:
- **Reference:** `training accuracy 98%; held-out accuracy 72%.`
- **Request:** `Describe the result in one sentence without claiming a cause.`
- **Response A:** `The model achieves 98% training accuracy and 72% held-out accuracy; the gap warrants further investigation.`
- **Response B:** `The model's excellent 98% held-out accuracy proves that regularization eliminated overfitting.`

Second, as shown on *Confidence and Calibration*, a response's assertive wording (`"proves"`, `"excellent"`) is neither a token likelihood nor a calibrated claim confidence. A set of predictions assigned confidence $p$ is **calibrated** if roughly a fraction $p$ of them are actually correct.

**Predict:**
1. Which specific substrings in Response B violate (a) factual fidelity to the reference and (b) the negative constraint (`"without claiming a cause"`)?
2. In the toy calibration table below, Model 1 and Model 2 make the exact same top-1 predictions across 100 items and have the exact same overall accuracy (`72%`). Which model has the larger Expected Calibration Error (ECE), and why?


In [6]:
# Part 1: Programmatic rubric check on the slide's two responses + an optional self-test Response C
reference = {"train_acc": "98%", "held_out_acc": "72%"}
responses = {
    "Response A": "The model achieves 98% training accuracy and 72% held-out accuracy; the gap warrants further investigation.",
    "Response B": "The model's excellent 98% held-out accuracy proves that regularization eliminated overfitting.",
    # Your Turn self-test: keeps both numbers accurate, but adds a causal explanation ("because it memorized...")
    "Response C (self-test)": "The model reaches 98% training accuracy and 72% held-out accuracy because it memorized noisy training examples.",
}

causal_claim_markers = ["proves that", "because", "caused by", "eliminated"]

print("=== Reference-Based Rubric Inspection ===")
for name, text in responses.items():
    has_train = "98% training accuracy" in text
    has_held_out = "72% held-out accuracy" in text
    swapped_metric = "98% held-out accuracy" in text
    causal_spans = [m for m in causal_claim_markers if m in text.lower()]
    passes = has_train and has_held_out and not swapped_metric and len(causal_spans) == 0
    print(f"\n{name}: {text!r}")
    print(f"  - Preserves 98% training & 72% held-out? {has_train and has_held_out} (swapped={'98% held-out accuracy' if swapped_metric else None})")
    print(f"  - Forbidden causal claim spans found:    {causal_spans}")
    print(f"  - Satisfies full rubric?                 {passes}")

# Part 2: Toy calibration comparison (bins of predicted confidence vs. observed correctness)
# 3 confidence bins, 100 predictions total
bin_counts = np.array([30, 40, 30])
observed_correct = np.array([18, 30, 24])  # empirical accuracies: 0.60, 0.75, 0.80 (overall 72%)

conf_calibrated = np.array([0.60, 0.75, 0.80])     # Model 1: well-calibrated confidence
conf_overconfident = np.array([0.85, 0.92, 0.98])  # Model 2: sharpened/overconfident confidence

acc_per_bin = observed_correct / bin_counts

def compute_ece(counts, accs, confs):
    weights = counts / counts.sum()
    gaps = np.abs(confs - accs)
    return np.sum(weights * gaps), gaps

ece_cal, gaps_cal = compute_ece(bin_counts, acc_per_bin, conf_calibrated)
ece_over, gaps_over = compute_ece(bin_counts, acc_per_bin, conf_overconfident)

print("\n=== Confidence vs. Observed Correctness Across Bins ===")
print(f"{'Bin':<5} {'Count':<7} {'Obs. Acc':<10} {'Model 1 Conf':<14} {'Gap 1':<8} {'Model 2 Conf':<14} {'Gap 2'}")
print("-" * 70)
for i in range(len(bin_counts)):
    print(
        f"{i+1:<5d} {bin_counts[i]:<7d} {acc_per_bin[i]:<10.2f} "
        f"{conf_calibrated[i]:<14.2f} {gaps_cal[i]:<8.2f} "
        f"{conf_overconfident[i]:<14.2f} {gaps_over[i]:.2f}"
    )

print(f"\nOverall accuracy for both models: {observed_correct.sum() / bin_counts.sum():.2%}")
print(f"Model 1 (Calibrated)    ECE: {ece_cal:.4f}")
print(f"Model 2 (Overconfident) ECE: {ece_over:.4f}")


=== Reference-Based Rubric Inspection ===

Response A: 'The model achieves 98% training accuracy and 72% held-out accuracy; the gap warrants further investigation.'
  - Preserves 98% training & 72% held-out? True (swapped=None)
  - Forbidden causal claim spans found:    []
  - Satisfies full rubric?                 True

Response B: "The model's excellent 98% held-out accuracy proves that regularization eliminated overfitting."
  - Preserves 98% training & 72% held-out? False (swapped=98% held-out accuracy)
  - Forbidden causal claim spans found:    ['proves that', 'eliminated']
  - Satisfies full rubric?                 False

Response C (self-test): 'The model reaches 98% training accuracy and 72% held-out accuracy because it memorized noisy training examples.'
  - Preserves 98% training & 72% held-out? True (swapped=None)
  - Forbidden causal claim spans found:    ['because']
  - Satisfies full rubric?                 False

=== Confidence vs. Observed Correctness Across Bins ===
Bi

**Compare with your prediction:** Check the detected failing spans for `Response B` and `Response C (self-test)`, and compare `Gap 1` vs. `Gap 2` in the calibration table.

<details>
<summary>Summary of results: Failing spans in Response B/C and accuracy (72%) vs. ECE (0.0000 vs. 0.1970)</summary>

1. **Rubric evidence:**
   - **Response A** preserves both reference numbers (`"98% training accuracy"` and `"72% held-out accuracy"`), states one sentence, and avoids claiming a cause (`"warrants further investigation"`).
   - **Response B** fails on both axes: it swaps the metric (`"98% held-out accuracy"`), invents `"regularization"`, and asserts a causal conclusion (`"proves that ... eliminated"`).
   - **Response C (self-test)** gets both numbers right (`98%` training, `72%` held-out) but still fails the instruction constraint because it asserts an unverified cause (`"because"`).
2. **Calibration vs. accuracy:** Both models achieve the exact same overall accuracy (`72.00%`) across the 100 items. However, Model 2 assigns confidences (`0.85`, `0.92`, `0.98`) that substantially exceed its empirical hit rates (`0.60`, `0.75`, `0.80`), yielding an Expected Calibration Error of `0.1970` compared to `0.0000` for Model 1.
</details>

**Check yourself.**

1. Why can blinded pairwise human preference (e.g., Chatbot Arena) rank an answer like Response B above a cautious answer when voters do not check a reference?
2. On the slide *Confidence and Calibration*, what three distinct quantities must not be confused when discussing an assistant's certainty?
3. Does the historical finding in the GPT-4 report (that post-training reduced multiple-choice calibration in their setup) imply that post-training always reduces factual accuracy?

<details>
<summary>Solution: Preference vs. factuality, three certainty quantities, and scoped calibration evidence</summary>

1. Without an explicit reference check, fluent and decisive prose (`"excellent"`, `"proves that"`) often looks helpful and authoritative to casual raters, illustrating why preference votes and factual correctness must be measured on separate evaluation axes.
2. (a) **Token probability:** next-token likelihood $p_\theta(y_t \mid x, y_{<t})$; (b) **Claim confidence:** the estimated probability that an entire statement or answer is correct; (c) **Response style:** how certain or assertive the wording sounds.
3. No. As the slide notes, accuracy and calibration are distinct properties (Model 1 and Model 2 above have identical 72% accuracy with very different calibration), and the GPT-4 report's calibration plot is a scoped empirical result on a specific multiple-choice benchmark rather than a universal law.
</details>
